# 112 — De modelo y automatización a agente

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

**Definición operativa:** agente = LLM que, en un bucle, decide qué acción ejecutar
(incluida terminar), observa el resultado real y usa esa observación para el siguiente
paso, al servicio de un objetivo verificable y bajo límites explícitos. Cuatro componentes
necesarios: objetivo, acciones, observación, bucle de decisión.

**Tres regímenes:**

- **Modelo:** una llamada, sin entorno.
- **Workflow:** grafo de pasos escrito por el ingeniero; el LLM rellena casillas.
- **Agente:** el control de flujo lo decide el modelo en cada iteración; la trayectoria
  emerge de la interacción con el entorno.

**Regla de ingeniería** (Anthropic, *Building effective agents*): usar la solución más
simple que resuelva la tarea. Agente solo cuando pasos y orden no se conocen a priori.


### 🎚️ Espectro de autonomía

```text
L0 modelo puro → L1 workflow con LLM → L2 router → L3 agente acotado (solo lectura
o con aprobación) → L4 agente con efectos + presupuesto/permisos → L5 autonomía extendida
```

A mayor autonomía, más superficie de fallo y más controles obligatorios.

El laboratorio `agent` ejecuta el caso mínimo: objetivo "verificar estado y sumar 7 + 5",
dos herramientas (`status`, `sum`), y una traza donde cada acción conserva argumentos y
observación. Su limitación declarada — "el plan es determinista" — señala la diferencia
con un agente LLM: aquí la política está cableada; allá la elige el modelo cada iteración.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** La traza tiene dos pasos: `status()` → `{"service": "demo",
"healthy": true}` verifica la primera condición; `sum(7, 5)` → `12` verifica la segunda.
Solo tras la segunda observación las dos condiciones del objetivo son verdaderas: recién
ahí el agente puede terminar en éxito. Terminar antes sería "parar por haber ejecutado
pasos", no por evidencia.

**Ejercicio 2.** (a) L0 — modelo puro, sin acciones. (b) L1 — workflow: el grafo lo
escribió el ingeniero. (c) L2 — router: elige una rama de un menú cerrado. (d) L4 acotado —
bucle libre con efectos (editar, ejecutar) bajo presupuesto de 20 iteraciones. (e) Fuera
de contrato: la misma capacidad sin límites no es "más autonomía", es ausencia de
contención (el nivel L4 exige presupuesto y permisos por definición).

**Ejercicio 3.** Versión operativa posible: "éxito ⇔ los 12 módulos de `docs/` compilan
sin warnings con `mkdocs build --strict` Y cada función pública de `src/` aparece en la
referencia de API; parar al cumplirse ambas o al agotar 30 pasos". La versión ambigua se
puede optimizar literalmente: añadir texto de relleno, reformatear sin corregir, o
declarar "mejorada" la documentación sin criterio — nada lo contradice.

**Ejercicio 4.** Traza de 3 pasos: (1) `status()` → `{"error": "timeout"}` → decisión:
reintentar (el objetivo no es verificable con un error); (2) `status()` →
`{"healthy": true}` → primera condición ✓; (3) `sum(7,5)` → `12` → segunda condición ✓,
terminar. Consume un paso extra del presupuesto. Para auditoría, cada entrada debería
registrar además la **decisión tomada tras la observación** (reintentar/continuar/parar)
y el motivo; la traza del laboratorio solo guarda acción y observación — suficiente para
reproducir, no para explicar.


In [ ]:
result = run_lab("agent", seed=112)
assert result["kind"] == "agent"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicio 1 — reconstrucción del estado del objetivo, verificada
result = run_lab("agent", seed=112)
objetivo = {"healthy": None, "sum": None}
for i, paso in enumerate(result["result"]["trace"], start=1):
    tool = paso["action"]["tool"]
    obs = paso["observation"]
    if tool == "status":
        objetivo["healthy"] = obs["healthy"]
    elif tool == "sum":
        objetivo["sum"] = obs
    completo = objetivo["healthy"] is True and objetivo["sum"] == 12
    print(f"paso {i}: {tool} -> objetivo={objetivo}  puede_terminar={completo}")

assert result["result"]["final"] == {"healthy": True, "sum": 12}


In [ ]:
# Ejercicio 4 — traza contrafactual con reintento (formato auditable)
traza_contrafactual = [
    {"action": {"tool": "status", "args": {}},
     "observation": {"error": "timeout"},
     "decision": "reintentar: sin observación válida no se verifica el objetivo"},
    {"action": {"tool": "status", "args": {}},
     "observation": {"service": "demo", "healthy": True},
     "decision": "continuar: condición healthy verificada"},
    {"action": {"tool": "sum", "args": {"left": 7, "right": 5}},
     "observation": 12,
     "decision": "terminar: ambas condiciones del objetivo verificadas"},
]
print(f"pasos consumidos: {len(traza_contrafactual)} (uno más que la traza sin fallo)")
# El campo extra 'decision' es lo que convierte la traza reproducible en explicable.


## Reflexión

1. La traza del laboratorio termina cuando `healthy == true` y `sum == 12`. ¿Qué haría el
   bucle si `status()` devolviera `healthy: false`, y por qué esa diferencia (parar por
   observación vs parar por "ya ejecuté mis pasos") es la frontera entre agente y script?
2. El laboratorio declara "el plan es determinista" como limitación. ¿Qué dos propiedades
   nuevas (una de capacidad, una de riesgo) aparecen cuando la política de decisión pasa
   de estar cableada a ser elegida por un LLM en cada iteración?
3. Da un ejemplo de tarea de tu entorno que HOY resolverías como workflow y el cambio de
   requisito mínimo que la convertiría en un problema de agente. ¿Qué límite (presupuesto,
   permiso o aprobación) añadirías en ese mismo momento?
